# 🔀 Parallel Execution Model: Supersteps, Joins, Reducers and `Send`

## Learning Objectives
In this notebook, you will learn:
1. **Supersteps and fan-out** - See which nodes LangGraph runs together, superstep by superstep, and check in code that parallel branches share one
2. **Joins** - Choose between separate edges, a list edge and `defer=True` when the branches into a node have different lengths
3. **Reducers** - Fix the `InvalidUpdateError` raised by parallel writes to one key, with `operator.add` or a dict-merge reducer
4. **Map-reduce with `Send`** - Fan out to *one branch per document*, however many documents there are

## Where this fits
Module 08's [`06_ensemble_parallel_exploration.ipynb`](../optional/04b_ensemble_parallel_exploration.ipynb) ran a committee of
analysts in parallel without taking the machinery apart. This notebook is that machinery. Parts 1-3 use plain-Python
stub nodes (no model, no waiting), so every result is exact and the same on every run; Part 4 maps an LLM over
documents. `02_async_multi_agent_streaming.ipynb` then makes the nodes `async`, measures the overlap with a stopwatch
and streams the run.

## Prerequisites
- Basic understanding of LangGraph `StateGraph`, nodes, and edges
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (only Part 4 calls a model)

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` (used only by Part 4) and imports the building blocks.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import operator
from typing import Annotated

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.errors import InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from typing_extensions import TypedDict

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY (Part 4 only) from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: used only by Part 4 (one summary per document)
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧱 Part 1: Supersteps and Fan-Out

LangGraph runs a graph in **supersteps**. In each superstep it runs every node that the previous superstep
triggered, waits for all of them, and only then applies their state updates and works out which nodes run next.
**Fan-out** is several edges out of one node: all their targets are triggered at once, so they land in the same
superstep and run in parallel.

The nodes in Parts 1-3 are stubs: plain functions that only append their own name to a `log` list. With no model
and no waiting, the superstep layout is the only thing being shown, and it is exact.

### Key Concepts:
- **Superstep**: one round of the graph; its nodes run in parallel, and their updates are applied together when it ends
- **Fan-out**: several `add_edge(source, node)` calls from one source put all the targets in the next superstep
- **`stream_mode="debug"`**: reports every node LangGraph starts, with the number of the superstep it belongs to

### 1.1 🧰 Stub Nodes and a Superstep Recorder

`stub(name)` builds a node that does no work. `run_by_superstep()` runs a graph once with
`stream_mode=["debug", "values"]`: the `debug` events say which superstep each node ran in, and the last `values`
event is the final state. `log` carries a reducer (`operator.add`) so that several nodes can append to it in the same
superstep; Part 3 shows why that is needed.

In [ ]:
# ============================================================================
# STUB NODES AND A SUPERSTEP RECORDER (no LLM)
# ============================================================================
class TraceState(TypedDict):
    log: Annotated[list[str], operator.add]  # every node appends its own name


def stub(name: str):
    """A node that does no work: it only appends its name to `log`."""

    def node(state: dict) -> dict:
        return {"log": [name]}

    return node


def run_by_superstep(graph, inputs: dict) -> tuple[dict, dict]:
    """Run `graph` once; return ({superstep: sorted names of the nodes that ran in it}, final state)."""
    steps, final = {}, None
    for mode, chunk in graph.stream(inputs, stream_mode=["debug", "values"]):
        if mode == "debug" and chunk["type"] == "task":  # one event per node LangGraph starts
            steps.setdefault(chunk["step"], []).append(chunk["payload"]["name"])
        elif mode == "values":  # the full state after each superstep; the last one is the final state
            final = chunk
    return {step: sorted(names) for step, names in steps.items()}, final


print("✅ Stub node factory and superstep recorder ready")

### 1.2 🌿 Fan-Out: One Planner, Three Searchers, One Report

`plan` fans out to three searchers, and every searcher has its own edge into `report`. The cell prints the nodes of
each superstep and checks in code that the three searchers share one.

In [ ]:
# ============================================================================
# FAN-OUT: one planner, three searchers, one report (stub nodes)
# ============================================================================
SEARCHERS = ["search_web", "search_docs", "search_db"]

builder = StateGraph(TraceState)
for name in ["plan", *SEARCHERS, "report"]:
    builder.add_node(name, stub(name))
builder.add_edge(START, "plan")
for name in SEARCHERS:
    builder.add_edge("plan", name)    # fan-out: three edges out of one node
    builder.add_edge(name, "report")  # every searcher feeds the report
builder.add_edge("report", END)
fan_out_graph = builder.compile()

steps, final = run_by_superstep(fan_out_graph, {"log": []})
for step, names in steps.items():
    print(f"superstep {step}: {', '.join(names)}")

step_of = {name: step for step, names in steps.items() for name in names}
same_step = len({step_of[name] for name in SEARCHERS}) == 1
report_once_after = final["log"].count("report") == 1 and step_of["report"] > step_of["search_web"]
print(f"\nThe three searchers share one superstep: {'✅' if same_step else '❌'}")
print(f"report ran once, in the superstep after them: {'✅' if report_once_after else '❌'}")

**Reading the output:** `plan` runs alone in the first superstep, the three searchers share the second, and `report`
runs once, in the third. Nodes of one superstep run in parallel: plain `def` nodes in a thread pool, `async def`
nodes on the event loop. This part proves which superstep each node belongs to, not overlap in time; module 08's
`06_ensemble_parallel_exploration.ipynb` measures the overlap for three sync LLM agents, and
`02_async_multi_agent_streaming.ipynb` puts a stopwatch on async ones.

`report` ran once here only because every branch is one step long. Part 2 shows what happens when they are not.

---
## 🔗 Part 2: Joins — Separate Edges, a List Edge, or `defer=True`

A **join** is a node with several incoming branches. With **separate edges** (`add_edge("a", "join")` and
`add_edge("b2", "join")`), the join runs once for every superstep in which one of its predecessors finished. When the
branches have different lengths, that is more than once, and the first run sees only the short branch's result.
Two ways make it wait for everything:
- **A list edge**, `add_edge(["a", "b2"], "join")`: the join runs once, in the superstep after *all* the listed nodes
  have finished.
- **`add_node("join", ..., defer=True)`**: the join is held back until the run has nothing else left to do, then
  runs once.

The demo builds one unequal-branch graph three ways: branch `a` is one step long, branch `b1 -> b2` is two. The
join counts its own runs in state (a counter with an `operator.add` reducer) and records which branch results it
could already read.

In [ ]:
# ============================================================================
# ONE UNEQUAL-BRANCH GRAPH, THREE JOIN STYLES (stub nodes)
# ============================================================================
class JoinState(TypedDict):
    log: Annotated[list[str], operator.add]
    join_runs: Annotated[int, operator.add]       # counter: every join run adds 1
    join_saw: Annotated[list[str], operator.add]  # the branch results each join run could read


def join(state: dict) -> dict:
    """The join records which branch results were already in the state when it ran."""
    ready = [name for name in ("a", "b2") if name in state["log"]]
    return {"log": ["join"], "join_runs": 1, "join_saw": [" + ".join(ready)]}


def build_join_graph(style: str):
    """START -> a (one step) and START -> b1 -> b2 (two steps), joined in the given style."""
    builder = StateGraph(JoinState)
    for name in ["a", "b1", "b2"]:
        builder.add_node(name, stub(name))
    builder.add_node("join", join, defer=(style == "defer=True"))
    builder.add_edge(START, "a")
    builder.add_edge(START, "b1")
    builder.add_edge("b1", "b2")
    if style == "list edge":
        builder.add_edge(["a", "b2"], "join")  # one edge from both: wait for all of them
    else:
        builder.add_edge("a", "join")   # separate edges: each one triggers the join on its own
        builder.add_edge("b2", "join")
    builder.add_edge("join", END)
    return builder.compile()


JOIN_STYLES = {"separate edges": 2, "list edge": 1, "defer=True": 1}  # style -> expected join runs
join_graphs = {style: build_join_graph(style) for style in JOIN_STYLES}
print(f"✅ Built {len(join_graphs)} join graphs: {list(join_graphs)}")

### 2.1 ▶️ How Many Times Does the Join Run?

Each graph runs once. The table shows the counted join runs against the expected count, and what each join run saw;
below it, the nodes of every superstep.

In [ ]:
# ============================================================================
# RUN THE THREE JOIN STYLES: count the join runs
# ============================================================================
print(f"{'join style':<16}{'join runs':>10}{'expected':>10}    what each join run saw")
timelines = {}
for style, expected in JOIN_STYLES.items():
    steps, final = run_by_superstep(join_graphs[style], {"log": [], "join_runs": 0, "join_saw": []})
    timelines[style] = " -> ".join("{" + ", ".join(names) + "}" for names in steps.values())
    verdict = "✅" if final["join_runs"] == expected else "❌"
    print(f"{style:<16}{final['join_runs']:>10}{expected:>10}  {verdict} {' | '.join(final['join_saw'])}")

print("\nNodes per superstep:")
for style, timeline in timelines.items():
    print(f"  {style:<16}{timeline}")

**Reading the output:** with separate edges, `join` ran twice. The first run shared a superstep with `b2`, when only
`a` had finished, so it saw `a` alone; the second run came after `b2` and saw `a + b2`. A real join (a synthesizer,
module 08's CIO) would have written one answer from half its inputs and then a second one. The list edge puts a
barrier in front of `join` that opens once both `a` and `b2` have finished. `defer=True` keeps the separate edges but
holds `join` back until nothing else is left to run. Both run it once, with everything.

**Which one to use:** a list edge when you know the predecessors at build time; module 08's ensemble uses one for its
CIO. It waits for every node it names, so if a conditional edge skips one of them on some run, the join never runs.
`defer=True` waits for the run to go quiet instead, so it also suits branches that are chosen at run time.

---
## ➕ Part 3: Reducers — Parallel Writes to One Key

Nodes of one superstep write their updates together when the superstep ends. If two of them write the **same key**
and the key has no reducer, LangGraph cannot know which value should win and raises `InvalidUpdateError`. A
**reducer**, the function in `Annotated[type, reducer]`, tells it how to combine the values instead:
- `Annotated[list, operator.add]` concatenates lists: every branch appends its items (the `log` key above).
- `Annotated[int, operator.add]` adds numbers: Part 2's `join_runs` counter.
- A dict-merge reducer such as `merge_dicts` below merges one-entry dicts into one dict keyed by branch.

A key that only one node per superstep writes needs no reducer.

The demo runs the same two finder nodes twice. Only the state schema changes.

In [ ]:
# ============================================================================
# NO REDUCER: two branches write one key in the same superstep
# ============================================================================
def finder(name: str):
    """A stub branch that writes the same two keys as every other finder."""

    def node(state: dict) -> dict:
        return {"findings": [f"{name}: result"], "by_finder": {name: "result"}}

    return node


def build_two_finders(state_schema):
    """START -> finder_a and START -> finder_b: both run, and write, in the same superstep."""
    builder = StateGraph(state_schema)
    for name in ["finder_a", "finder_b"]:
        builder.add_node(name, finder(name))
        builder.add_edge(START, name)
        builder.add_edge(name, END)
    return builder.compile()


class NoReducerState(TypedDict):
    findings: list[str]  # no reducer: at most one write per superstep
    by_finder: dict


try:
    build_two_finders(NoReducerState).invoke({"findings": [], "by_finder": {}})
    raised = False
except InvalidUpdateError as error:
    raised = True
    print(f"InvalidUpdateError: {str(error).splitlines()[0]}")
print(f"Two branches writing one plain key raise InvalidUpdateError: {'✅' if raised else '❌'}")

### 3.1 🧩 The Same Two Branches, With Reducers

`findings` gets `operator.add`; `by_finder` gets `merge_dicts`, a reducer that merges the old dict with the new one.

In [ ]:
# ============================================================================
# WITH REDUCERS: operator.add for a list, merge_dicts for a dict
# ============================================================================
def merge_dicts(left: dict, right: dict) -> dict:
    """Reducer: combine parallel updates to one dict-valued key."""
    return {**(left or {}), **(right or {})}


class ReducerState(TypedDict):
    findings: Annotated[list[str], operator.add]  # each branch's list is appended
    by_finder: Annotated[dict, merge_dicts]       # each branch's one-entry dict is merged in


merged = build_two_finders(ReducerState).invoke({"findings": [], "by_finder": {}})
print(f"findings  (operator.add): {merged['findings']}")
print(f"by_finder (merge_dicts):  {merged['by_finder']}")

both_kept = merged["findings"] == ["finder_a: result", "finder_b: result"] and set(merged["by_finder"]) == {"finder_a", "finder_b"}
print(f"\nBoth branches' writes kept, no error: {'✅' if both_kept else '❌'}")

**Reading the output:** the same two nodes fail or succeed depending only on the state schema. The combined list
comes out in the same order on every run, whichever branch finishes first: LangGraph applies a superstep's writes in
a fixed order when the superstep ends, not in finish order. For dicts, `operator.or_` (`left | right`) does the same
job as `merge_dicts`; module 08's ensemble uses it for its analysts' reports.

---
## 🗺️ Part 4: Map-Reduce — Parallel Document Summarization

The **map-reduce** pattern applies the same operation independently to each item ("map") and then combines the
results ("reduce"). The number of documents is only known at run time, so fixed edges cannot express the fan-out.
LangGraph's **`Send`** API does: a routing function returns one `Send("summarize_one", {...})` per document, and
LangGraph runs all of those branches in the same superstep.

### Key Concepts:
- **`Send(node, payload)`**: run `node` with its own private input `payload`
- **Conditional edge returning a list of `Send`s**: dynamic fan-out
- **`operator.add` reducer** on `summaries`: every branch appends its summary

In [ ]:
# ============================================================================
# MAP-REDUCE STATE AND GRAPH (dynamic fan-out with Send)
# ============================================================================
class MapReduceState(TypedDict):
    documents: list[str]
    summaries: Annotated[list[str], operator.add]  # each map branch appends one summary
    final_summary: str


class DocState(TypedDict):
    index: int
    document: str


def fan_out(state: MapReduceState) -> list[Send]:
    """One Send per document: LangGraph runs every summarize_one branch in the same step."""
    return [Send("summarize_one", {"index": i, "document": d}) for i, d in enumerate(state["documents"])]


def summarize_one(state: DocState) -> dict:
    """The map step for ONE document (runs once per Send, in parallel)."""
    response = llm.invoke(
        [SystemMessage(content="Summarize this document in 2-3 sentences."), HumanMessage(content=state["document"])]
    )
    return {"summaries": [f"[doc {state['index'] + 1}] {response.content}"]}


def reduce_combine(state: MapReduceState) -> dict:
    """The reduce step: combine all summaries."""
    response = llm.invoke(
        [
            SystemMessage(content="Combine these summaries into one coherent overview."),
            HumanMessage(content="\n\n".join(sorted(state["summaries"]))),
        ]
    )
    return {"final_summary": response.content}


graph = StateGraph(MapReduceState)
graph.add_node("summarize_one", summarize_one)
graph.add_node("reduce", reduce_combine)
graph.add_conditional_edges(START, fan_out, ["summarize_one"])
graph.add_edge("summarize_one", "reduce")
graph.add_edge("reduce", END)
map_reduce = graph.compile()

print("✅ Map-reduce graph compiled. Edges:")
for edge in map_reduce.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}")

### 4.1 ▶️ Running the Map-Reduce Demo

`stream_mode="updates"` reports each map branch as it finishes, so you can see one update per document before the
single `reduce` update.

In [ ]:
# ============================================================================
# RUN THE MAP-REDUCE DEMO
# ============================================================================
documents = [
    "Python is a high-level programming language known for its simplicity and readability. It supports multiple programming paradigms and has a vast ecosystem of libraries.",
    "Machine learning is a subset of AI that enables systems to learn from data. Common approaches include supervised, unsupervised, and reinforcement learning.",
    "Cloud computing provides on-demand access to computing resources. Major providers include AWS, Azure, and Google Cloud Platform.",
]

map_updates = 0
for update in map_reduce.stream({"documents": documents, "summaries": []}, stream_mode="updates"):
    for node, payload in update.items():
        if node == "summarize_one":
            map_updates += 1
            print(f"  map branch finished: {payload['summaries'][0][:90]}...")
        else:
            print(f"\n{node} finished.\n\nCombined summary:\n{payload['final_summary']}")

print(f"\nDocuments: {len(documents)} | map branches run: {map_updates} {'✅' if map_updates == len(documents) else '❌'}")

**Reading the output:** there is one `map branch finished` line per document, and the counter at the end confirms
it. Add a fourth document and a fourth branch appears, with no change to the graph. That is what `Send` adds over
fixed edges. The summaries may finish in any order, so `reduce_combine` sorts them by their `[doc N]` tag first.

---
## 📝 Summary

In this notebook, we learned:

### 1. Supersteps and Fan-Out
- LangGraph runs a graph in **supersteps**: the nodes of one superstep run in parallel, and their updates are applied together when it ends
- Several edges out of one node put all their targets in the next superstep; `stream_mode="debug"` shows which superstep each node ran in

### 2. Joins
- With **separate edges**, a join runs once per superstep in which a predecessor finished: twice for unequal branches, the first time with half the inputs
- A **list edge** (`add_edge(["a", "b2"], "join")`) waits for every listed node; **`defer=True`** waits until nothing else is left to run. Both run the join once

### 3. Reducers
- Two nodes writing one plain key in the same superstep raise `InvalidUpdateError`
- A reducer combines the writes: `operator.add` for lists and counters, a dict merge (`merge_dicts`, `operator.or_`) for dicts; a key only one node writes needs none

### 4. Map-Reduce with `Send`
- **Map**: a conditional edge returns one `Send` per item, so the fan-out size is decided at run time
- **Reduce**: `operator.add` collects every branch's summary; one node combines them

### Next Steps
- `02_async_multi_agent_streaming.ipynb`: async nodes, `ainvoke`, and `astream` to watch concurrent agents work, with a stopwatch on the overlap